In [1]:
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

def convert_to_30fps(video_path):
    """Đưa video về 30 FPS, tự giảm hoặc nội suy nếu cần."""
    tmp_path = video_path + ".tmp.mp4"
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-filter:v", "fps=fps=30",
        "-an",
        "-c:v", "libx264",
        "-preset", "ultrafast",
        tmp_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.replace(tmp_path, video_path)
    print(f"{os.path.basename(video_path)} → 30fps")
    return True


def process_dataset(root="Data", max_workers=8):
    """Duyệt toàn bộ dataset và đồng bộ tất cả video về 30fps."""
    all_videos = []
    for subdir, _, files in os.walk(root):
        for f in files:
            if f.endswith(".mp4"):
                all_videos.append(os.path.join(subdir, f))

    print(f"Tổng video cần xử lý: {len(all_videos)}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(convert_to_30fps, v) for v in all_videos]
        for f in as_completed(futures):
            f.result()

    print(f"\nToàn bộ video trong '{root}' đã đồng bộ về 30fps.")

if __name__ == "__main__":
    root = input("Nhập thư mục chứa video: ").strip()
    process_dataset(root, max_workers=8)


Tổng video cần xử lý: 3000
4.mp4 → 30fps
2.mp4 → 30fps
1.mp4 → 30fps
0.mp4 → 30fps
5.mp4 → 30fps
6.mp4 → 30fps
3.mp4 → 30fps
7.mp4 → 30fps
8.mp4 → 30fps
0.mp4 → 30fps
2.mp4 → 30fps
9.mp4 → 30fps
1.mp4 → 30fps
3.mp4 → 30fps
4.mp4 → 30fps
5.mp4 → 30fps
6.mp4 → 30fps
8.mp4 → 30fps
7.mp4 → 30fps
0.mp4 → 30fps
2.mp4 → 30fps
9.mp4 → 30fps
1.mp4 → 30fps
3.mp4 → 30fps
4.mp4 → 30fps
6.mp4 → 30fps
5.mp4 → 30fps
8.mp4 → 30fps
0.mp4 → 30fps
7.mp4 → 30fps
9.mp4 → 30fps
1.mp4 → 30fps
2.mp4 → 30fps
4.mp4 → 30fps
6.mp4 → 30fps
3.mp4 → 30fps
5.mp4 → 30fps
7.mp4 → 30fps
8.mp4 → 30fps
0.mp4 → 30fps
1.mp4 → 30fps
2.mp4 → 30fps
3.mp4 → 30fps
9.mp4 → 30fps
4.mp4 → 30fps
6.mp4 → 30fps
5.mp4 → 30fps
8.mp4 → 30fps
7.mp4 → 30fps
9.mp4 → 30fps
0.mp4 → 30fps
2.mp4 → 30fps
1.mp4 → 30fps
4.mp4 → 30fps
6.mp4 → 30fps
3.mp4 → 30fps
0.mp4 → 30fps
5.mp4 → 30fps
1.mp4 → 30fps
8.mp4 → 30fps
5.mp4 → 30fps
2.mp4 → 30fps
7.mp4 → 30fps
3.mp4 → 30fps
4.mp4 → 30fps
8.mp4 → 30fps
6.mp4 → 30fps
9.mp4 → 30fps
7.mp4 → 30fps
9.mp4 →

In [3]:
import numpy as np
import os
import cv2

def get_frame_stats(folder):
    lengths = []
    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)
        if not os.path.isdir(cls_path): continue
        for f in os.listdir(cls_path):
            if f.endswith(".mp4"):
                cap = cv2.VideoCapture(os.path.join(cls_path, f))
                lengths.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
                cap.release()
    print(f"Trung bình: {np.mean(lengths):.1f} frames | Trung vị: {np.median(lengths):.1f} | Max: {np.max(lengths)}")
    return lengths

lengths = get_frame_stats("Data/train")
print(np.percentile(lengths, 95))
max_frames = min( int(np.percentile(lengths, 95)), 160 )
print(max_frames)

Trung bình: 100.0 frames | Trung vị: 91.0 | Max: 332
179.04999999999973
160
